In [19]:
import os 
os.environ['SPARK_HOME'] = "/Users/mukesh/opt/spark-3.5.1-bin-hadoop3"
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
# os.environ['PATH'] is usually handled by these two, but setting it explicitly can't hurt:
os.environ['PATH'] = os.environ['SPARK_HOME'] + "/bin:" + os.environ.get('PATH', '')

In [20]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 

from mysql_spark import ConnectDB

In [21]:
HOST = "127.0.0.1"
USER = "root"
PASSWORD = "Paridhi@2019#"  
DATABASE = "dw_poc"
SOURCE_TABLE = "source_data"
TARGET_TABLE = "processed_results"





In [22]:
db = ConnectDB(host=HOST, password=PASSWORD, user=USER, database=DATABASE)

Initialising the database configuration...


In [23]:
## THIS IS THE WORKAROUND TO LOAD MYSQL CONNECTOR JAR IN SPARK

from pyspark.sql import SparkSession
import os
import glob

# Build a list of likely search roots relative to the notebook process cwd
cwd = os.getcwd()
search_roots = [
    cwd,
    os.path.join(cwd, 'jars'),
    os.path.abspath(os.path.join(cwd, '..')),
    os.path.abspath(os.path.join(cwd, '..', 'jars')),
    os.path.abspath(os.path.join(cwd, '..', '..')),
    os.path.abspath(os.path.join(cwd, '..', '..', 'jars')),
    # explicit project root (helpful if kernel cwd is SCD/)
    os.path.abspath(os.path.join(os.path.expanduser('~'), 'Desktop', 'TEST_AGAIN', 'jars'))
]

patterns = ['*mysql*connector*.jar', 'mysql-connector*.jar', '*mysql*.jar']

matches = []
for root in search_roots:
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(root, '**', pat), recursive=True))

# Also fallback: global recursive search under repo root (limited depth)
repo_root = os.path.abspath(os.path.join(cwd, '..'))
matches.extend(glob.glob(os.path.join(repo_root, '**', '*mysql*connector*.jar'), recursive=True))

matches = sorted(set(matches))
mysql_jar = matches[0] if matches else None

if mysql_jar:
    print(f"Found MySQL JDBC driver jar: {mysql_jar}")
    spark = (
        SparkSession.builder.appName("SCD")
        .config("spark.jars", mysql_jar)
        .config("spark.driver.extraClassPath", mysql_jar)
        .getOrCreate()
    )
    print('Configured Spark with jar on driver and executors.')
else:
    print("No MySQL JDBC driver jar found in expected locations. Looked in:")
    for r in search_roots:
        print(' -', r)
    print('You can place the connector jar in one of those folders (e.g. ./jars) and restart the kernel.')
    spark = SparkSession.builder.appName("SCD").getOrCreate()

print("Spark-Version *** :", spark.version)


Found MySQL JDBC driver jar: /Users/mukesh/Desktop/TEST_AGAIN/jars/mysql-connector-j-9.4.0.jar
Configured Spark with jar on driver and executors.
Spark-Version *** : 3.5.1


### ETL Logic

In [5]:
HIGH_DATE = lit("9999-12-31").cast(DateType())
TODAY = lit("2025-01-20").cast(DateType()) # Mock the effective date for incoming data


In [24]:
# Create SparkSession

from pyspark.sql import SparkSession

spark = (
        SparkSession.builder.appName("SCD2")
        .config("spark.jars", mysql_jar)
        .config("spark.driver.extraClassPath", mysql_jar)
        .getOrCreate()
    )

25/11/04 08:32:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


#### INSERT  DATA  INTO  SOURCE  &  TARGET  FOR  THE  INITIAL  SNAPSHOT 

In [25]:
# We have out table in mysql , need to insert some records for simulation 
from datetime import date


truncate_src = "truncate table employee_src"
truncate_trg = "truncate table employee_trg"
db.execute_query(truncate_src)
db.execute_query(truncate_trg)


query1 = "insert into employee_src values(1,'MUKESH','HR',5000,'2025-10-30')"
query2 = "insert into employee_src values(2,'YASH','IT',6000,'2025-10-30')"
query3 = "insert into employee_src values(3,'CHARLIE','FINANCE',8000,'2025-10-30')"

SRC = [query1, query2, query3]
for query in SRC:
    db.execute_query(query)

tar1 = f"insert into employee_trg values (1,'Mukesh','HR',5000,'{date(2025,9,1)}','{date(9999,12,31)}',True)"
tar2 = f"insert into employee_trg values (2,'Bob','IT',6000,'{date(2025,9,15)}','{date(9999,12,31)}',True)"
tar3 = f"insert into employee_trg values (3,'Charlie','Finance',15000,'{date(2025,9,10)}','{date(9999,12,31)}',True)"
           
TRG = [tar1, tar2, tar3]
for query in TRG:
    db.execute_query(query)




Standard MySQL Connection Established.
Standard Query executed successfully. Rows affected: 0
Standard Query executed successfully. Rows affected: 0
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1


In [26]:
# Read data from source table into Spark DataFrame

df_src = db.read_mysql_table_to_spark_df(spark, 'employee_src')

df_src.printSchema()

df_src.show()



--- Spark Read: Reading Entire Table 'employee_src' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

+------+-------+----------+------+--------------+
|emp_id|   name|department|salary|effective_date|
+------+-------+----------+------+--------------+
|     1| MUKESH|        HR|  5000|    2025-10-30|
|     2|   YASH|        IT|  6000|    2025-10-30|
|     3|CHARLIE|   FINANCE|  8000|    2025-10-30|
+------+-------+----------+------+--------------+



In [27]:
# READ TARGET TABLE INTO A DATAFRAME 


df_trg = db.read_mysql_table_to_spark_df(spark, 'employee_trg')

df_trg.printSchema()

df_trg.show()


--- Spark Read: Reading Entire Table 'employee_trg' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------------+------------------+----+
|     1| Mukesh|        HR|  5000|          2025-0

In [28]:
join_df = df_src.alias("src").join(df_trg.alias("trg"),on="emp_id",how="left")

join_df.printSchema()

join_df.show()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+-------+----------+------+--------------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_date|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------+-------+----------+------+--------------------+------------------+----+
|     1| MUKESH|        HR|  5000|    2025-10-30| Mukesh|        HR|  5000|          2025-09-01|        9999-12-31|true|
|     3|CHARLIE|   FINANCE|  8000|    2025-10-30|Charlie| 

In [29]:
changed_df = join_df.filter(
          (col("trg.emp_id").isNull()) | (col("src.name") != col("trg.name"))
).select("src.*")

changed_df.show()

+------+-------+----------+------+--------------+
|emp_id|   name|department|salary|effective_date|
+------+-------+----------+------+--------------+
|     1| MUKESH|        HR|  5000|    2025-10-30|
|     3|CHARLIE|   FINANCE|  8000|    2025-10-30|
|     2|   YASH|        IT|  6000|    2025-10-30|
+------+-------+----------+------+--------------+



In [30]:
df_src.printSchema()
df_trg.printSchema()



root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)



In [31]:
# Separate Target into Active and Historical Components ---

df_target_active = df_trg.filter(col("flag") == True).alias("T")
df_target_historical = df_trg.filter(col("flag") == False)


df_target_active.show()
df_target_historical.show()



+------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------------+------------------+----+
|     1| Mukesh|        HR|  5000|          2025-09-01|        9999-12-31|true|
|     2|    Bob|        IT|  6000|          2025-09-15|        9999-12-31|true|
|     3|Charlie|   Finance| 15000|          2025-09-10|        9999-12-31|true|
+------+-------+----------+------+--------------------+------------------+----+

+------+----+----------+------+--------------------+------------------+----+
|emp_id|name|department|salary|effective_start_date|effective_end_date|flag|
+------+----+----------+------+--------------------+------------------+----+
+------+----+----------+------+--------------------+------------------+----+



In [32]:
# Compare Active Records in Target (T) with Source (S) ---

# We use a FULL OUTER JOIN to find all matches (updates) and non-matches (new/deleted).
# Join condition: Match on the key (emp_id)

df_join = df_target_active.join(df_src.alias("S"), col("T.emp_id") == col("S.emp_id"), "fullouter")

df_join.show()

# Define the condition that signifies a change in any tracked attribute (SCD Type 2 trigger)
CHANGE_DETECTED = (
    (col("T.name") != col("S.name")) |
    (col("T.department") != col("S.department")) |
    (col("T.salary") != col("S.salary"))
)

+------+-------+----------+------+--------------------+------------------+----+------+-------+----------+------+--------------+
|emp_id|   name|department|salary|effective_start_date|effective_end_date|flag|emp_id|   name|department|salary|effective_date|
+------+-------+----------+------+--------------------+------------------+----+------+-------+----------+------+--------------+
|     1| Mukesh|        HR|  5000|          2025-09-01|        9999-12-31|true|     1| MUKESH|        HR|  5000|    2025-10-30|
|     2|    Bob|        IT|  6000|          2025-09-15|        9999-12-31|true|     2|   YASH|        IT|  6000|    2025-10-30|
|     3|Charlie|   Finance| 15000|          2025-09-10|        9999-12-31|true|     3|CHARLIE|   FINANCE|  8000|    2025-10-30|
+------+-------+----------+------+--------------------+------------------+----+------+-------+----------+------+--------------+



In [33]:
# df_expired : Records that need to be EXPIRED (Match found AND change detected)
# Logic : If a match is found (emp_id exists in both T and S) AND any tracked attribute has changed,
# then we need to expire the existing active record in Target by setting its effective_end_date to
# the day before the new record's effective_start_date.


df_expired = df_join.filter(col("T.emp_id").isNotNull() & col("S.emp_id").isNotNull() & CHANGE_DETECTED) \
    .select(
        col("T.emp_id"),
        col("T.name"),
        col("T.department"),
        col("T.salary"),
        col("T.effective_start_date"),
        date_sub(TODAY, 1).alias("effective_end_date"), # End date is day before new effective date
        lit(False).alias("flag") # Set flag to False (Expired)
    )

df_expired.show()

+------+-------+----------+------+--------------------+------------------+-----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date| flag|
+------+-------+----------+------+--------------------+------------------+-----+
|     1| Mukesh|        HR|  5000|          2025-09-01|        2025-01-19|false|
|     2|    Bob|        IT|  6000|          2025-09-15|        2025-01-19|false|
|     3|Charlie|   Finance| 15000|          2025-09-10|        2025-01-19|false|
+------+-------+----------+------+--------------------+------------------+-----+



In [34]:
# df_unchanged :: Records that are UNCHANGED (Match found AND NO change detected)

df_unchanged = df_join.filter(col("T.emp_id").isNotNull() & col("S.emp_id").isNotNull() & ~CHANGE_DETECTED) \
    .select(
        col("T.emp_id"),
        col("T.name"),
        col("T.department"),
        col("T.salary"),
        col("T.effective_start_date"),
        col("T.effective_end_date"),
        col("T.flag")
    )

df_unchanged.show()

+------+----+----------+------+--------------------+------------------+----+
|emp_id|name|department|salary|effective_start_date|effective_end_date|flag|
+------+----+----------+------+--------------------+------------------+----+
+------+----+----------+------+--------------------+------------------+----+



In [35]:
# df_new_insert : Records that are NEW VERSIONS or NEW EMPLOYEES (All source records)
# We only need to filter out records that are already up-to-date and unchanged (101)

df_new_insert = df_join.filter(
        # Records with changes (expired in step A) OR brand new source records
        CHANGE_DETECTED | col("T.emp_id").isNull()
    ).select(
        col("S.emp_id"),
        col("S.name"),
        col("S.department"),
        col("S.salary"),
        TODAY.alias("effective_start_date"), # New version starts today
        HIGH_DATE.alias("effective_end_date"),
        lit(True).alias("flag")
    ).distinct() # Distinct to remove duplicates from the join logic

df_new_insert.show()

+------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------------+------------------+----+
|     3|CHARLIE|   FINANCE|  8000|          2025-01-20|        9999-12-31|true|
|     2|   YASH|        IT|  6000|          2025-01-20|        9999-12-31|true|
|     1| MUKESH|        HR|  5000|          2025-01-20|        9999-12-31|true|
+------+-------+----------+------+--------------------+------------------+----+



In [36]:
# --- 6. Final Union and Overwrite ---

# Union all components:
# 1. Previous historical records (must be kept)
# 2. Expired records (the old version of updated employees)
# 3. Unchanged active records (employees who are current and did not update)
# 4. New records (new employees and new versions of updated employees)
df_final = df_target_historical \
    .unionByName(df_expired) \
    .unionByName(df_unchanged) \
    .unionByName(df_new_insert) \
    .orderBy("emp_id", "effective_start_date")

df_final.show()


+------+-------+----------+------+--------------------+------------------+-----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date| flag|
+------+-------+----------+------+--------------------+------------------+-----+
|     1| MUKESH|        HR|  5000|          2025-01-20|        9999-12-31| true|
|     1| Mukesh|        HR|  5000|          2025-09-01|        2025-01-19|false|
|     2|   YASH|        IT|  6000|          2025-01-20|        9999-12-31| true|
|     2|    Bob|        IT|  6000|          2025-09-15|        2025-01-19|false|
|     3|CHARLIE|   FINANCE|  8000|          2025-01-20|        9999-12-31| true|
|     3|Charlie|   Finance| 15000|          2025-09-10|        2025-01-19|false|
+------+-------+----------+------+--------------------+------------------+-----+



In [37]:

db.write_spark_data_to_mysql(df_final, "scd_test", "append")

# --- 7. Show Final Result ---
print("\n--- FINAL SCD TYPE 2 RESULT (Pure PySpark Join/Union/Overwrite) ---")
#spark.table(TARGET_TABLE_NAME + "_FINAL").show(truncate=False)

spark.stop()


--- Spark Write: Writing to 'scd_test' in 'append' mode ---
Successfully wrote Spark DataFrame to MySQL table 'scd_test'.

--- FINAL SCD TYPE 2 RESULT (Pure PySpark Join/Union/Overwrite) ---
